# Домашнее задание 20. Классификация организации облаков (Understanding Cloud Organization)

**Задача:** по спутниковым снимкам сегментировать 4 типа организации неглубоких облаков — `Fish`, `Flower`, `Gravel`, `Sugar`. Каждое изображение может содержать несколько типов одновременно (multi-label semantic segmentation), поэтому задача решается как бинарная сегментация по 4 независимым каналам.

**Данные:** Kaggle competition *Understanding Cloud Organization* (скачаны по прямой ссылке Google Drive, т.к. официальный Kaggle API требует авторизации/принятия правил соревнования).

**План:**
1. EDA данных и масок
2. Dataset / DataLoader с аугментациями
3. Архитектура сети — компактная U-Net
4. Dice loss / Dice coefficient как метрика качества
5. Обучение с ранней остановкой (early stopping) по Dice на валидации
6. Графики обучения и визуальная оценка предсказаний

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from sklearn.model_selection import train_test_split

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type == 'cuda':
    print(torch.cuda.get_device_name(0))

RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

DATA_DIR = 'data'
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images')
CLASSES = ['Fish', 'Flower', 'Gravel', 'Sugar']

## 1. EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
train_df.head(10)

In [ ]:
train_df.shape

Формат `train.csv`: одна строка на пару (изображение, класс). `Image_Label` = `<имя_файла>_<класс>`, `EncodedPixels` — маска в формате RLE (run-length encoding) или NaN, если данный класс отсутствует на снимке.

In [ ]:
train_df['ImageId'] = train_df['Image_Label'].apply(lambda x: x.split('_')[0])
train_df['ClassName'] = train_df['Image_Label'].apply(lambda x: x.split('_')[1])
train_df['HasMask'] = train_df['EncodedPixels'].notna().astype(int)

n_images = train_df['ImageId'].nunique()
print(f'Уникальных изображений: {n_images}')
print(f'Строк в разметке: {len(train_df)} (= {n_images} изображений x {len(CLASSES)} класса)')

In [ ]:
class_counts = train_df.groupby('ClassName')['HasMask'].sum().reindex(CLASSES)
plt.figure(figsize=(7, 4))
class_counts.plot(kind='bar', color='#4C72B0')
plt.title('Число изображений с маской по каждому классу облаков')
plt.ylabel('Количество изображений')
plt.xticks(rotation=0)
plt.show()
class_counts

In [ ]:
labels_per_image = train_df.groupby('ImageId')['HasMask'].sum()
plt.figure(figsize=(7, 4))
labels_per_image.value_counts().sort_index().plot(kind='bar', color='#55A868')
plt.title('Сколько типов облаков одновременно присутствует на одном снимке')
plt.xlabel('Число классов на изображении')
plt.ylabel('Количество изображений')
plt.xticks(rotation=0)
plt.show()

Почти на каждом снимке присутствует несколько типов облачной организации одновременно (обычно 1-3 из 4), поэтому это задача **multi-label сегментации**, а не классификации одного класса на изображение.

### 1.1 Функции для работы с RLE-масками

In [ ]:
def rle_decode(mask_rle, shape):
    if not isinstance(mask_rle, str) or mask_rle == '':
        return np.zeros(shape, dtype=np.uint8)
    s = mask_rle.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0:][::2], s[1:][::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(shape[0] * shape[1], dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape(shape, order='F')


def build_mask(image_id, df, shape):
    """Возвращает маску (H, W, 4) — по каналу на класс облаков."""
    mask = np.zeros((*shape, len(CLASSES)), dtype=np.uint8)
    rows = df[df['ImageId'] == image_id]
    for _, row in rows.iterrows():
        class_idx = CLASSES.index(row['ClassName'])
        mask[:, :, class_idx] = rle_decode(row['EncodedPixels'], shape)
    return mask

### 1.2 Примеры изображений с масками

In [ ]:
ORIG_SHAPE = (1400, 2100)  # (H, W) исходных снимков

sample_ids = train_df['ImageId'].drop_duplicates().sample(3, random_state=RANDOM_STATE).tolist()
colors = [(255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0)]

fig, axes = plt.subplots(len(sample_ids), 1, figsize=(12, 5 * len(sample_ids)))
for ax, img_id in zip(axes, sample_ids):
    img = np.array(Image.open(os.path.join(TRAIN_IMG_DIR, img_id)).convert('RGB'))
    mask = build_mask(img_id, train_df, img.shape[:2])
    overlay = img.copy()
    for c in range(len(CLASSES)):
        color_layer = np.zeros_like(img)
        color_layer[:, :] = colors[c]
        alpha = (mask[:, :, c] * 0.35)[:, :, None]
        overlay = (overlay * (1 - alpha) + color_layer * alpha).astype(np.uint8)
    ax.imshow(overlay)
    present = [CLASSES[c] for c in range(len(CLASSES)) if mask[:, :, c].sum() > 0]
    ax.set_title(f'{img_id} | классы: {present}')
    ax.axis('off')
plt.tight_layout()
plt.show()

Границы между типами облачной организации нечёткие и плавные — это видно на визуализации: маски классов часто накладываются или граничат друг с другом без резкого перепада, что типично для этой задачи и объясняет, почему она сложна для классических rule-based алгоритмов.

## 2. Dataset и DataLoader

Изображения в датасете имеют высокое разрешение (1400×2100), что на GPU с 4GB видеопамяти (Quadro P2000) неподъёмно для полноразмерного обучения. Уменьшаем разрешение до **320×480** (кратно 32, чтобы корректно работал downsampling/upsampling в U-Net) — это стандартная практика для этого соревнования.

In [ ]:
IMG_HEIGHT, IMG_WIDTH = 320, 480

train_mean = (0.485, 0.456, 0.406)
train_std = (0.229, 0.224, 0.225)

train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=train_mean, std=train_std),
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=train_mean, std=train_std),
])

In [ ]:
class CloudDataset(Dataset):
    def __init__(self, image_ids, df, img_dir, transform=None):
        self.image_ids = image_ids
        self.df = df
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img = np.array(Image.open(os.path.join(self.img_dir, image_id)).convert('RGB'))
        mask = build_mask(image_id, self.df, img.shape[:2])

        if self.transform is not None:
            augmented = self.transform(image=img, mask=mask)
            img, mask = augmented['image'], augmented['mask']

        img = torch.from_numpy(img.transpose(2, 0, 1)).float()
        mask = torch.from_numpy(mask.transpose(2, 0, 1)).float()
        return img, mask

### 2.1 Train / Validation split

Чтобы обучение на CPU/скромной GPU занимало разумное время, ограничим выборку подмножеством изображений (достаточным, чтобы модель могла осмысленно обучиться и была видна динамика на графиках), с делением на train/val.

In [ ]:
N_SAMPLES = 1200  # часть полного датасета (5546 изображений) для разумного времени обучения

all_ids = train_df['ImageId'].drop_duplicates().tolist()
sample_ids_all = pd.Series(all_ids).sample(n=min(N_SAMPLES, len(all_ids)), random_state=RANDOM_STATE).tolist()

train_ids, val_ids = train_test_split(sample_ids_all, test_size=0.2, random_state=RANDOM_STATE)
print(f'train: {len(train_ids)}, val: {len(val_ids)}')

In [ ]:
train_dataset = CloudDataset(train_ids, train_df, TRAIN_IMG_DIR, transform=train_transform)
val_dataset = CloudDataset(val_ids, train_df, TRAIN_IMG_DIR, transform=val_transform)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

xb, yb = next(iter(train_loader))
print('batch image shape:', xb.shape, 'batch mask shape:', yb.shape)

## 3. Архитектура сети — компактная U-Net

Используем классическую U-Net архитектуру (энкодер-декодер со skip-connections), но с уменьшенным числом каналов (`base=16`), чтобы модель уверенно помещалась в 4GB видеопамяти при разумном batch size, сохраняя при этом характерные для сегментации свойства: свёрточные блоки, downsampling пулингом, upsampling транспонированными свёртками и конкатенацию признаков энкодера с декодером.

In [ ]:
class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=4, base=16):
        super().__init__()
        self.enc1 = ConvBlock(in_ch, base)
        self.enc2 = ConvBlock(base, base * 2)
        self.enc3 = ConvBlock(base * 2, base * 4)
        self.enc4 = ConvBlock(base * 4, base * 8)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(base * 8, base * 16)

        self.up4 = nn.ConvTranspose2d(base * 16, base * 8, 2, stride=2)
        self.dec4 = ConvBlock(base * 16, base * 8)
        self.up3 = nn.ConvTranspose2d(base * 8, base * 4, 2, stride=2)
        self.dec3 = ConvBlock(base * 8, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = ConvBlock(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = ConvBlock(base * 2, base)

        self.out_conv = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b = self.bottleneck(self.pool(e4))

        d4 = self.up4(b)
        d4 = self.dec4(torch.cat([d4, e4], dim=1))
        d3 = self.up3(d4)
        d3 = self.dec3(torch.cat([d3, e3], dim=1))
        d2 = self.up2(d3)
        d2 = self.dec2(torch.cat([d2, e2], dim=1))
        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1))

        return self.out_conv(d1)


model = UNet(in_ch=3, out_ch=len(CLASSES), base=16).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'Параметров в модели: {n_params:,}')

## 4. Dice loss и Dice coefficient

**Dice coefficient** — основная метрика качества сегментации в этой задаче: $$Dice = \frac{2 |X \cap Y|}{|X| + |Y|}$$
где $X$ — предсказанная маска, $Y$ — истинная. Значение 1 означает идеальное совпадение, 0 — полное несовпадение. Dice особенно хорошо подходит для задач с сильным дисбалансом классов (облако vs фон), в отличие от pixel accuracy.

В качестве функции потерь используем комбинацию `BCEWithLogitsLoss` (стабильный градиент на старте обучения) и `DiceLoss` (напрямую оптимизирует целевую метрику) — так модель одновременно учится и попиксельной классификации, и корректной геометрии масок.

In [ ]:
def dice_coefficient(pred_logits, target, threshold=0.5, eps=1e-7):
    pred = (torch.sigmoid(pred_logits) > threshold).float()
    intersection = (pred * target).sum(dim=(2, 3))
    union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
    dice = (2 * intersection + eps) / (union + eps)
    return dice.mean()


class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps

    def forward(self, pred_logits, target):
        pred = torch.sigmoid(pred_logits)
        intersection = (pred * target).sum(dim=(2, 3))
        union = pred.sum(dim=(2, 3)) + target.sum(dim=(2, 3))
        dice = (2 * intersection + self.eps) / (union + self.eps)
        return 1 - dice.mean()


class BCEDiceLoss(nn.Module):
    def __init__(self, bce_weight=0.5):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss()
        self.dice = DiceLoss()
        self.bce_weight = bce_weight

    def forward(self, pred_logits, target):
        return self.bce_weight * self.bce(pred_logits, target) + (1 - self.bce_weight) * self.dice(pred_logits, target)

## 5. Early stopping

In [ ]:
import copy

class EarlyStopping:
    """Останавливает обучение, если Dice на валидации не улучшается patience эпох подряд."""
    def __init__(self, patience=5, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_score = None
        self.counter = 0
        self.should_stop = False
        self.best_state = None

    def step(self, score, model):
        if self.best_score is None or score > self.best_score + self.min_delta:
            self.best_score = score
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

## 6. Обучение

In [ ]:
N_EPOCHS = 30
LR = 1e-3
PATIENCE = 5

model = UNet(in_ch=3, out_ch=len(CLASSES), base=16).to(device)
criterion = BCEDiceLoss(bce_weight=0.5)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)
early_stopping = EarlyStopping(patience=PATIENCE)

history = {'train_loss': [], 'val_loss': [], 'train_dice': [], 'val_dice': []}

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, total_dice, n_batches = 0.0, 0.0, 0
    torch.set_grad_enabled(is_train)
    for imgs, masks in loader:
        imgs, masks = imgs.to(device), masks.to(device)

        if is_train:
            optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)

        if is_train:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        total_dice += dice_coefficient(outputs, masks).item()
        n_batches += 1

    torch.set_grad_enabled(True)
    return total_loss / n_batches, total_dice / n_batches

In [ ]:
import time

for epoch in range(1, N_EPOCHS + 1):
    t0 = time.time()
    train_loss, train_dice = run_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_dice = run_epoch(model, val_loader, criterion, optimizer=None)
    scheduler.step(val_dice)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_dice'].append(train_dice)
    history['val_dice'].append(val_dice)

    early_stopping.step(val_dice, model)
    elapsed = time.time() - t0
    print(f'Epoch {epoch:02d}/{N_EPOCHS} | '
          f'train_loss={train_loss:.4f} val_loss={val_loss:.4f} | '
          f'train_dice={train_dice:.4f} val_dice={val_dice:.4f} | '
          f'{elapsed:.1f}s | patience={early_stopping.counter}/{PATIENCE}')

    if early_stopping.should_stop:
        print(f'\nRanняя остановка на эпохе {epoch}: Dice на валидации не улучшался {PATIENCE} эпох подряд.')
        break

model.load_state_dict(early_stopping.best_state)
print(f'\nЛучший Dice на валидации: {early_stopping.best_score:.4f}')

### 6.1 График обучения

In [ ]:
epochs_range = range(1, len(history['train_loss']) + 1)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, history['train_loss'], marker='o', label='train')
axes[0].plot(epochs_range, history['val_loss'], marker='o', label='val')
axes[0].set_title('Loss (BCE + Dice)')
axes[0].set_xlabel('Эпоха')
axes[0].set_ylabel('Loss')
axes[0].legend()

axes[1].plot(epochs_range, history['train_dice'], marker='o', label='train')
axes[1].plot(epochs_range, history['val_dice'], marker='o', label='val')
axes[1].axvline(len(epochs_range) - early_stopping.counter, color='red', linestyle='--', alpha=0.5, label='лучшая эпоха')
axes[1].set_title('Dice coefficient')
axes[1].set_xlabel('Эпоха')
axes[1].set_ylabel('Dice')
axes[1].legend()

plt.tight_layout()
plt.show()

## 7. Оценка качества и визуализация предсказаний

In [ ]:
per_class_dice = {c: [] for c in CLASSES}
model.eval()
with torch.no_grad():
    for imgs, masks in val_loader:
        imgs, masks = imgs.to(device), masks.to(device)
        outputs = model(imgs)
        preds = (torch.sigmoid(outputs) > 0.5).float()
        for c in range(len(CLASSES)):
            intersection = (preds[:, c] * masks[:, c]).sum(dim=(1, 2))
            union = preds[:, c].sum(dim=(1, 2)) + masks[:, c].sum(dim=(1, 2))
            dice_c = (2 * intersection + 1e-7) / (union + 1e-7)
            per_class_dice[CLASSES[c]].extend(dice_c.cpu().numpy().tolist())

dice_summary = pd.Series({c: np.mean(v) for c, v in per_class_dice.items()}).round(4)
print('Средний Dice по классам на валидации:')
dice_summary

In [ ]:
plt.figure(figsize=(7, 4))
dice_summary.plot(kind='bar', color='#C44E52')
plt.title('Средний Dice coefficient по классам (validation)')
plt.ylabel('Dice')
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.show()

### 7.1 Визуальное сравнение предсказаний с истинными масками

In [ ]:
def denormalize(img_tensor, mean=train_mean, std=train_std):
    img = img_tensor.cpu().numpy().transpose(1, 2, 0)
    img = img * np.array(std) + np.array(mean)
    return np.clip(img, 0, 1)


model.eval()
imgs, masks = next(iter(val_loader))
with torch.no_grad():
    outputs = model(imgs.to(device))
    preds = (torch.sigmoid(outputs) > 0.5).float().cpu()

n_show = 3
fig, axes = plt.subplots(n_show, 3, figsize=(15, 5 * n_show))
for i in range(n_show):
    img_vis = denormalize(imgs[i])
    axes[i, 0].imshow(img_vis)
    axes[i, 0].set_title('Изображение')
    axes[i, 0].axis('off')

    true_mask = masks[i].sum(dim=0).numpy()
    axes[i, 1].imshow(img_vis)
    axes[i, 1].imshow(true_mask, alpha=0.5, cmap='jet')
    axes[i, 1].set_title('Истинная маска')
    axes[i, 1].axis('off')

    pred_mask = preds[i].sum(dim=0).numpy()
    axes[i, 2].imshow(img_vis)
    axes[i, 2].imshow(pred_mask, alpha=0.5, cmap='jet')
    axes[i, 2].set_title('Предсказанная маска')
    axes[i, 2].axis('off')
plt.tight_layout()
plt.show()

## 8. Выводы

1. Построена и обучена компактная U-Net (~1.9M параметров) для multi-label сегментации 4 типов облачной организации (`Fish`, `Flower`, `Gravel`, `Sugar`) на спутниковых снимках.
2. В качестве функции потерь использована комбинация BCE + Dice loss, а основной метрикой качества выступил **Dice coefficient** — он устойчив к сильному дисбалансу классов (облако/фон), в отличие от pixel accuracy.
3. Обучение остановлено алгоритмом **early stopping**, отслеживающим Dice на валидации: при отсутствии улучшения в течение нескольких эпох подряд обучение прекращается, а в модель возвращаются веса лучшей эпохи — это защищает от переобучения и экономит время.
4. Границы между классами облаков на исходных изображениях нечёткие (что отмечено в EDA) — соответствующие классы (например, `Gravel`/`Sugar`) обычно оказываются сложнее для модели, чем визуально более выраженные структуры (`Flower`), что отражается в различии Dice по классам.
5. Дальнейшие пути улучшения качества: использовать предобученный энкодер (например, ResNet, transfer learning), увеличить разрешение и объём обучающей выборки при доступности более мощного GPU, добавить TTA (test-time augmentation) и подбор порога бинаризации по классам.